# Three ways to draw 64 dimensions, and what each one throws away

MichAl Academy, lesson 2.13.

Run each cell with **Shift+Enter**.

Sixty-four features is already too many to look at, and every dataset in the
second half of this track has more. So the first move is to squash it into two
dimensions and look at it, which every course tells you to do and few of them
tell you what it costs.

This notebook measures the cost three ways. Then it runs the same methods on
pure noise, and the plot looks about as convincing as the real one.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from scipy.spatial.distance import pdist
from scipy.stats import pearsonr
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

# Colab: !pip install umap-learn
SEED = 0
digits = load_digits()
X_full = StandardScaler().fit_transform(digits.data)
y_full = digits.target
n_full = len(X_full)
print(f"{n_full} images, {X_full.shape[1]} features, {len(set(y_full))} classes")


## 1. PCA is arithmetic, and you can check it

PCA finds the directions along which the data varies most, keeps the first `k`
of them, and throws the rest away. The library reports how much variance each
direction holds.

That report is checkable. If you reconstruct the data from `k` components, the
squared error you are left with should be exactly the variance you discarded.


In [ ]:
pca_full = PCA(random_state=SEED).fit(X_full)
eigen = pca_full.explained_variance_

print(f"{'k':>4} {'variance kept':>14} {'reconstruction MSE':>20} {'discarded':>11} {'difference':>12}")
for k in (2, 5, 10, 20, 40):
    p = PCA(n_components=k, random_state=SEED).fit(X_full)
    mse = float(((X_full - p.inverse_transform(p.transform(X_full))) ** 2).sum(axis=1).mean())
    # sklearn's explained_variance_ uses the unbiased divisor n-1, and the MSE
    # above divides by n. Correct for it and the two agree exactly.
    discarded = float(eigen[k:].sum()) * (n_full - 1) / n_full
    print(f"{k:>4} {p.explained_variance_ratio_.sum():>14.4f} {mse:>20.6f}"
          f" {discarded:>11.6f} {abs(mse - discarded):>12.2e}")


Agreement to about 1e-14, which is floating point saying yes.

Note what had to be corrected to get there. `explained_variance_` divides by
n - 1, the unbiased estimator, and a mean squared error divides by n. Without
that factor the two agree to three decimals and look "approximately right",
which is the kind of nearly-right that hides real bugs. Reading a library's
divisor conventions is not pedantry.

And read the first row properly, because it is the number that matters for
every scatter plot you are about to make: **two components hold 21.6% of the
variance.** Whatever you see in a 2d PCA plot of this data, four fifths of it
is not on the page.


## 2. What each method preserves

Three questions, one per column, and the three methods answer them differently
because they were built to.

- **Distance.** Correlate every pair's distance in 2d against the same pair's
  distance in 64d.
- **Neighbourhood.** Of each point's ten nearest neighbours in 64d, how many
  are still among its ten nearest in 2d?
- **Class separation.** How well separated are the true digit classes in the
  projection? The methods never see the labels, so this is a fair test of
  whether the picture shows what is there.


In [ ]:
sub = np.random.default_rng(SEED).choice(n_full, 500, replace=False)
X, y = X_full[sub], y_full[sub]
K = 10


def neighbours(A, k=K):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(A)
    return nn.kneighbors(A, return_distance=False)[:, 1:]


base_nn = neighbours(X)
base_d = pdist(X)

embeddings = {
    "PCA 2d": PCA(n_components=2, random_state=SEED).fit_transform(X),
    "t-SNE 2d": TSNE(n_components=2, random_state=SEED, init="pca",
                     perplexity=30).fit_transform(X),
    "UMAP 2d": umap.UMAP(n_components=2, random_state=SEED,
                         n_neighbors=15).fit_transform(X),
    "PCA 10d": PCA(n_components=10, random_state=SEED).fit_transform(X),
}

print(f"{'method':<12} {'distance':>9} {'neighbours kept':>17} {'class separation':>18}")
print(f"{'the raw 64d':<12} {1.0:>9.4f} {f'{K} of {K}':>17} {silhouette_score(X, y):>18.4f}")
for name, e in embeddings.items():
    got = neighbours(e)
    kept = np.mean([len(set(a) & set(b)) for a, b in zip(base_nn, got)])
    print(f"{name:<12} {pearsonr(base_d, pdist(e)).statistic:>9.4f}"
          f" {f'{kept:.2f} of {K}':>17} {silhouette_score(e, y):>18.4f}")
print(f"\nchance, for the neighbours column: {K / (len(X) - 1) * K:.2f} of {K}")


Read it a column at a time.

**Distance.** PCA 0.6456, t-SNE 0.2255, UMAP 0.1873. PCA is a linear
projection so it keeps a fair amount; the other two keep little. **Do not read
distance off a t-SNE or UMAP plot.** The gaps between the blobs are not
measurements, and neither are the sizes of the blobs.

**Neighbourhood.** PCA 2.49 of 10, t-SNE 6.19, UMAP 5.84, against 0.20 for
chance. Exactly the reverse ordering. Neither family is defective; this is the
trade each one advertises. PCA promises variance, the neighbour embeddings
promise neighbourhoods, and both keep their promise.

**Class separation.** The raw 64d space scores 0.0850. t-SNE scores 0.3734 and
UMAP 0.4780, four and five times higher. Sit with that for a moment: **the
projections make the classes more visible than they are in the full data.**
These are not lossy summaries of a dataset, they are instruments tuned to show
neighbourhood structure, and discarding distance is how they do it.

Then look at the last row, which is the practical one. **PCA at ten dimensions
keeps neighbourhoods better than t-SNE at two**, 6.28 against 6.19, while also
keeping 0.9560 of the distance. If you want components to feed another model
rather than a picture to look at, that is the cheap, deterministic option, and
it has no perplexity to choose.


## 3. Now run it on nothing

Five hundred rows of independent Gaussian noise, sixty-four columns, no
structure of any kind. Project it and ask k-means how many clusters it can find.


In [ ]:
noise = np.random.default_rng(SEED).normal(size=(500, 64))


def apparent_clusters(e, up_to=15):
    best, score = None, -1.0
    for k in range(2, up_to):
        lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(e)
        s = silhouette_score(e, lab)
        if s > score:
            best, score = k, s
    return best, score


print(f"{'':<28} {'clusters':>9} {'silhouette':>11}")
for label, e in (
    ("t-SNE on real digits", embeddings["t-SNE 2d"]),
    ("UMAP on real digits", embeddings["UMAP 2d"]),
    ("t-SNE on pure noise", TSNE(n_components=2, random_state=SEED, init="pca",
                                 perplexity=30).fit_transform(noise)),
    ("UMAP on pure noise", umap.UMAP(n_components=2, random_state=SEED,
                                     n_neighbors=15).fit_transform(noise)),
    ("PCA on pure noise", PCA(n_components=2, random_state=SEED).fit_transform(noise)),
):
    k, s = apparent_clusters(e)
    print(f"{label:<28} {k:>9} {s:>11.4f}")


There it is. **Pure noise reaches a silhouette of about 0.35** in every one of
these projections, and under t-SNE and UMAP k-means confidently reports three
clusters in it.

Compare across the table rather than down it. UMAP on noise scores 0.3741. PCA
on the real digits scores 0.3847. Those are the same number for practical
purposes, and one of the two datasets has no structure in it at all.

Real digits reach 0.5309 under t-SNE. Higher, and the gap is real, but not so
much higher that 0.35 sits comfortably below any threshold you would have set
by eye. So:

**"I ran t-SNE and saw clusters" is not evidence of clusters.** Not because the
method is bad, but because a two-dimensional embedding of anything at all
produces lumps, and human eyes and k-means both find them.


In [ ]:
# Is 0.35 on noise a fluke of one seed? Five different noise draws.
scores = []
for seed in range(5):
    nz = np.random.default_rng(seed).normal(size=(500, 64))
    e = TSNE(n_components=2, random_state=SEED, init="pca", perplexity=30).fit_transform(nz)
    k, s = apparent_clusters(e)
    scores.append(s)
    print(f"noise seed {seed}: {k} clusters, silhouette {s:.4f}")
print(f"\nrange {min(scores):.4f} to {max(scores):.4f}, so it is not a fluke")


## 4. The test that does separate them

Silhouette on the projection cannot tell noise from digits reliably, 0.35
against 0.53. But look back at the neighbours column from section 2 and try the
same measurement on noise.


In [ ]:
print(f"{'':<28} {'neighbours kept':>17}")
for label, A in (("real digits", X), ("pure noise", noise)):
    b = neighbours(A)
    e = TSNE(n_components=2, random_state=SEED, init="pca", perplexity=30).fit_transform(A)
    got = neighbours(e)
    kept = np.mean([len(set(p) & set(q)) for p, q in zip(b, got)])
    print(f"t-SNE on {label:<19} {f'{kept:.2f} of {K}':>17}")


6.19 against 2.14, so roughly three times, on data where the silhouette of the
embedding gave only 0.5309 against 0.3443.

The reason is worth understanding rather than memorising. Noise has no
neighbourhood structure to preserve, so t-SNE cannot preserve any, and the
retention score collapses. But it can still produce lumps, so the silhouette
does not.

So on these two datasets, **neighbour retention separated real structure from
noise better than the silhouette of the embedding did.** That is a measurement
on two datasets rather than a theorem, and it is cheap enough to run on yours.


## 5. And the parameter with no principled default

t-SNE's perplexity is roughly how many neighbours each point should pay
attention to. There is no way to compute the right value.


In [ ]:
print(f"{'perplexity':>11} {'class separation':>17} {'apparent clusters':>18}")
for perp in (5, 30, 100, 300):
    e = TSNE(n_components=2, random_state=SEED, init="pca", perplexity=perp).fit_transform(X)
    k, _ = apparent_clusters(e)
    print(f"{perp:>11} {silhouette_score(e, y):>17.4f} {k:>18}")


Class separation runs from 0.1947 to 0.4233, a factor of two, on identical
data. Apparent clusters run from 12 to 14 where the truth is 10, and the best
separation comes from perplexity 5, which is the setting furthest from the
library default of 30.

Which means a t-SNE picture is a picture of your data **and** of a parameter you
chose. Run two or three perplexities before believing any of them, and if the
conclusion changes, the conclusion was about the parameter.


## What to take from this

| Claim | What we measured |
|---|---|
| Explained variance is a library number you trust | It equals the reconstruction error exactly, once you fix the n-1 divisor |
| A 2d PCA plot shows you the data | Two components held 21.6% of this data's variance |
| Distance in a t-SNE plot means something | Correlation with the real distances was 0.2255, and 0.1873 for UMAP |
| These methods lose information | They trade distance for neighbourhoods. Class separation went from 0.0850 raw to 0.4780 |
| Clusters in a t-SNE plot are clusters | Pure noise scores 0.3443 and shows 3 of them, repeatably across seeds |
| You need t-SNE to preserve neighbourhoods | PCA at 10 dimensions kept 6.28 of 10 against t-SNE's 6.19 |
| Perplexity is a detail | It moved class separation from 0.1947 to 0.4233 on the same data |

Three habits:

1. Print the variance kept before showing anybody a PCA plot.
2. Run the same projection on shuffled or synthetic noise before believing any
   structure in it.
3. If you need components for a model rather than a picture, try PCA first. It
   is deterministic, has no perplexity, and at ten dimensions it beat t-SNE's
   neighbour retention here while keeping 96% of the distance.


## Try this

1. Shuffle each column of the digits data independently, which destroys the
   relationships between features but keeps every column's distribution. Run
   t-SNE. This is a better null than Gaussian noise, and a better test.
2. Set `n_neighbors=2` and then `n_neighbors=200` in UMAP. It has the same
   problem perplexity does, under a different name.
3. Feed `PCA(n_components=10)` output to the random forest from lesson 2.5 and
   compare accuracy against the raw 64 features. Dimensionality reduction is
   usually sold as a visualisation tool; check whether it also pays as a
   preprocessing step here.
